## Refresh EPA American Indian Reservations Data

In [ ]:
import os,sys,psycopg2,tempfile,git;
from contextlib import closing;

sys.path.append(os.path.join(os.path.expanduser('~'),'notebooks'));
import common;

lddk = os.path.join(os.path.expanduser('~'),'loading_dock');
src  = 'https://services.arcgis.com/cJ9YHowT8TU7DUyn/ArcGIS/rest/services/American_Indian_Reservations__DOC___EPA_2023_/FeatureServer/1/query';

dbse = os.environ['POSTGRESQL_DB'];
host = os.environ['POSTGRESQL_HOST'];
port = os.environ['POSTGRESQL_PORT'];
user = 'cipsrv';
pswd = os.environ['POSTGRESQL_CIP_PASS'];
thds = 2;


In [ ]:
gith = os.environ['GITHUB_REPO_URL'];
# override: "git@github.com:USEPA/CIPv2.git"
brnh = os.environ['GITHUB_DEFAULT_BRANCH'];
# override: "mybranch"
brnh = 'owld_stubs'
td = tempfile.TemporaryDirectory();
    
repo = git.Repo.clone_from(
     url     = gith
    ,branch  = brnh
    ,to_path = td.name
    ,depth   = 1
);
sha = repo.head.commit.hexsha;
short_sha = repo.git.rev_parse(sha,short=7);

print("checkout of " + td.name + " complete.");

trg = os.path.join(td.name,'src','database','cipdev_support','materialized views');


In [ ]:
cs = "dbname=%s user=%s password=%s host=%s port=%s" % (
     dbse
    ,user
    ,pswd
    ,host
    ,port
);

try:
    conn = psycopg2.connect(cs);
except:
    raise Exception("database connection error");

print("database is ready");

In [ ]:
with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_support.epa_segs_air CASCADE");
    conn.commit();

print("any preexisting data purged.");

In [ ]:
cmd = '-overwrite -f PostgreSQL '                                                      \
    + 'postgresql://' + user + ':' + pswd + '@' + host + ':' + port + '/' + dbse + ' ' \
    + '"' + src + '?where=shape%20is%20not%20null&outFields=*&f=json" '                                    \
    + '-nln cipdev_support.epa_segs_air ';

cmd = cmd                                                                              \
    + '-t_srs EPSG:4269 '                                                              \
    + '-lco GEOMETRY_NAME=shape '                                                      \
    + '-lco GEOM_TYPE=geometry '                                                       \
    + '-lco LAUNDER=YES '                                                              \
    + '-lco DIM=2 '                                                                    \
    + '-lco SPATIAL_INDEX=GIST '                                                       \
    + '-nlt GEOMETRY';

z = common.ogr2ogr(
     cmdstring = cmd
);
print(str(z[1]));

In [ ]:
for item in ['5070','3338','26904','32161','32655','32702']:
    z = common.load_sqlfile(
         conn
        ,os.path.join(trg,'epa_segs_air_' + item + '.sql')
        ,echo=True
    );

In [ ]:
for item in ['m','h']:
    for item2 in ['5070','3338','26904','32161','32655','32702']:
        z = common.load_sqlfile(
             conn
            ,os.path.join(trg,'epa_segs_air_flat_' + item2 + '_' + item + '.sql')
            ,echo=True
        );

In [ ]:
for item in ['m','h']:
    for item2 in ['5070','3338','26904','32161','32655','32702']:
        z = common.load_sqlfile(
             conn
            ,os.path.join(trg,'epageofab_' + item + '_catchment_istribal_' + item2 + '.sql')
            ,echo=True
        );

In [ ]:
for item in ['m','h']:
    z = common.load_sqlfile(
         conn
        ,os.path.join(trg,'epageofab_' + item + '_catchment_istribal_final.sql')
        ,echo=True
    );

In [ ]:
with closing(conn.cursor()) as cursor:
    cursor.execute("GRANT SELECT ON cipdev_support.epageofab_m_catchment_istribal_final TO PUBLIC");
    cursor.execute("GRANT SELECT ON cipdev_support.epageofab_h_catchment_istribal_final TO PUBLIC");
    conn.commit();

print("refresh complete");
